# Notebook 0 - Conversion des données brutes en Parquet

Les fichiers Excel sont lourds (jusqu'à 98 Mo) et lents à lire (~1 minute chacun).
On les convertit **une seule fois** au format Parquet : les notebooks suivants se lancent alors en quelques secondes.

**Fichiers sources** (dossier `data/brut/`) :
* par défaut, les fichiers **Excel** (`.xlsx`) ;


In [1]:
import os
import re
import zipfile
import pandas as pd

fichiers = {
    "entreprises": "data/brut/entreprises_pseudonymisees",
    "travailleurs": "data/brut/travailleurs_pseudonymises",
    "sinistres": "data/brut/sinistres_pseudonymises",
}

## 1. Lecture et conversion

Pour chaque table : lecture du fichier Excel, puis écriture en Parquet dans `data/`.

In [2]:
for nom, chemin in fichiers.items():
    source = chemin + ".xlsx"
    df = pd.read_excel(source, engine="calamine")

    # les colonnes texte sont typées "string" pour que Parquet les enregistre correctement
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].astype("string")

    df.to_parquet(f"data/{nom}.parquet", index=False)
    print(f"{nom:13s} : {len(df):>9,} lignes lues depuis {source}")

entreprises   :   337,137 lignes lues depuis data/brut/entreprises_pseudonymisees.xlsx


travailleurs  : 1,048,001 lignes lues depuis data/brut/travailleurs_pseudonymises.xlsx


sinistres     :   321,127 lignes lues depuis data/brut/sinistres_pseudonymises.xlsx


## 2. Vérification : toutes les lignes ont-elles été chargées ?

Un fichier Excel indique lui-même sa taille (balise `dimension`, par exemple `A1:L1048002`). On compare ce nombre au nombre de lignes réellement lues : les totaux doivent **coïncider exactement**, ce qui confirme que l'extraction transmise par la CNPS est **complète**.

In [3]:
for nom, chemin in fichiers.items():
    df = pd.read_parquet(f"data/{nom}.parquet")

    if os.path.exists(chemin + ".csv"):
        print(f"{nom:13s} : {len(df):>9,} lignes (source CSV, pas de limite de lignes)")
        continue

    # nombre de lignes déclaré dans le fichier Excel (moins la ligne d'en-tête)
    with zipfile.ZipFile(chemin + ".xlsx") as z:
        debut = z.open("xl/worksheets/sheet1.xml").read(3000).decode()
    lignes_excel = int(re.search(r'ref="A1:[A-Z]+(\d+)"', debut).group(1)) - 1

    statut = "OK, tout est chargé" if lignes_excel == len(df) else "ERREUR : lignes manquantes"
    print(f"{nom:13s} : {len(df):>9,} lignes lues / {lignes_excel:>9,} dans l'Excel -> {statut}")


entreprises   :   337,137 lignes lues /   337,137 dans l'Excel -> OK, tout est chargé


travailleurs  : 1,048,001 lignes lues / 1,048,001 dans l'Excel -> OK, tout est chargé
sinistres     :   321,127 lignes lues /   321,127 dans l'Excel -> OK, tout est chargé
